# Tập 3.8 · Kết hợp cảm biến và theo dõi nhiều người 👥

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap38/tap38_theo_doi.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 15](https://porabook.com/book/sensor-fusion/)

Camera + radar ghép lại; Kalman + cổng Mahalanobis + Hungary theo dõi 5 người. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** phép ghép cảm biến, khoảng cách Mahalanobis và cập nhật track (3 bài); ô chấm chạy lại cả bộ theo dõi.
3. Vẽ track của hai cách so sánh.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap38")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import theo_doi as goc             # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"RMSE: chỉ camera {goc.RMSE_CB['cam']:.2f} m · chỉ radar {goc.RMSE_CB['rad']:.2f} m · ghép {goc.RMSE_CB['ghep']:.2f} m")
for ten, d in (("Gần nhất, không dự đoán", goc.DIEM_NN), ("Kalman + cổng + Hungary", goc.DIEM_KF)):
    print(f"{ten:24s}: đổi ID {d['IDSW']:3d} · FP {d['FP']:4d} · FN {d['FN']:4d} · MOTA {d['MOTA'] * 100:5.1f}%")

## 2. Bài tập

Ghép: $Y = \sum R_i^{-1}$, $\Sigma = Y^{-1}$, $z = \Sigma\sum R_i^{-1} z_i$. Mahalanobis: $m^2 = \tilde z^T S^{-1}\tilde z$, $S = HPH^T + R$. Cập nhật: $K = PH^TS^{-1}$, $x \mathrel{+}= K\tilde z$, $P = (I - KH)P$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `ghep(ds)` | ghép các cặp `(z, R)` |
| **2** | `m2(x, P, z, R)` | khoảng cách Mahalanobis bình phương |
| **3** | `cap_nhat(x, P, z, R)` | cập nhật Kalman một track |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

H = np.array([[1.0, 0, 0, 0], [0, 1, 0, 0]])


def ghep(ds):
    """Bài 1 (Mục 15.2.1, dạng thông tin): ghép nhiều phát hiện của cùng một người từ các cảm biến khác nhau.

        Y = Σ R_i⁻¹,   y = Σ R_i⁻¹·z_i,   Σ_ghép = Y⁻¹,   z_ghép = Σ_ghép·y
    ds là list các cặp (z, R). Trả về (z_ghep, S_ghep)."""
    raise NotImplementedError("TODO Bài 1: ghép theo dạng thông tin")


def m2(x, P, z, R):
    """Bài 2 (Mục 15.5.2): bình phương khoảng cách Mahalanobis giữa phát hiện (z, R) và track (x, P).

        S = H·P·Hᵀ + R,   z̃ = z − H·x,   m² = z̃ᵀ·S⁻¹·z̃
    Phát hiện chỉ được ghép với track nếu m² ≤ 9,21 (cổng χ² 2 bậc tự do, 99%). Trả về m² (số)."""
    raise NotImplementedError("TODO Bài 2: Mahalanobis")


def cap_nhat(x, P, z, R):
    """Bài 3: cập nhật Kalman một track bằng phát hiện đã ghép.

        S = H·P·Hᵀ + R,  K = P·Hᵀ·S⁻¹,  x_mới = x + K·(z − H·x),  P_mới = (I − K·H)·P
    Trả về (x_moi, P_moi)."""
    raise NotImplementedError("TODO Bài 3: cập nhật track")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 2–3, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Cổng rộng hay hẹp thì tốt? Đổi ngưỡng cổng rồi chấm MOTA.

In [ ]:
for gate in [2.0, 5.99, 9.21, 25.0]:
    goc.GATE = gate
    d = goc.cham_diem(goc.chay_kalman_gnn())
    print(f"cổng m² ≤ {gate:5.2f}: đổi ID {d['IDSW']:2d} · FP {d['FP']:3d} · FN {d['FN']:3d} · MOTA {d['MOTA'] * 100:5.1f}%")
goc.GATE = 9.21

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap38).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)